In [1]:
import pandas as pd
import torch

from transformers import AutoTokenizer
from transformers import AutoModelForSequenceClassification
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
import numpy as np
from tqdm import tqdm

d:\PPTI 15 ARTEMIS\Semester 8 (Skripsi)\TAM UTAUT\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
MODEL_NAME = "indobenchmark/indobert-base-p2"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

In [3]:
sentiment_model = AutoModelForSequenceClassification.from_pretrained(
    "models/sentiment_indobert"
)

sentiment_model.eval()

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 558.38it/s, Materializing param=classifier.weight]                                      


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(50000, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

In [4]:
utaut_model = AutoModelForSequenceClassification.from_pretrained(
    "models/utaut_indobert"
)

utaut_model.eval()

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 470.23it/s, Materializing param=classifier.weight]                                      


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(50000, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

In [5]:
sentiment_map = {
    0:"negative",
    1:"neutral",
    2:"positive"
}

In [6]:
def batch_predict(texts, batch_size=32):

    sentiments = []
    PE_list = []
    EE_list = []
    SI_list = []
    FC_list = []

    for i in tqdm(range(0, len(texts), batch_size)):

        batch_texts = texts[i:i+batch_size]

        inputs = tokenizer(
            batch_texts,
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=128
        )

        with torch.no_grad():

            sentiment_logits = sentiment_model(**inputs).logits
            utaut_logits = utaut_model(**inputs).logits

        # =====================
        # SENTIMENT
        # =====================

        sentiment_ids = sentiment_logits.argmax(dim=1).cpu().numpy()
        sentiment_labels = [sentiment_map[i] for i in sentiment_ids]

        sentiments.extend(sentiment_labels)

        # =====================
        # UTAUT
        # =====================

        utaut_probs = torch.sigmoid(utaut_logits).cpu().numpy()

        multiplier = np.select(
            [
                utaut_probs <= 0.48,
                utaut_probs <= 0.59,
                utaut_probs <= 0.69,
                utaut_probs <= 0.79,
                utaut_probs <= 0.89,
                utaut_probs <= 1.00
            ],
            [0, 1, 2, 3, 4, 5]
        )

        # probabilitas × multiplier
        utaut_scores = utaut_probs * multiplier

        # pembulatan (≥0.5 ke atas, <0.5 ke bawah)
        utaut_scores = np.round(utaut_scores).astype(int)

        # simpan hasil
        PE_list.extend(utaut_scores[:,0])
        EE_list.extend(utaut_scores[:,1])
        SI_list.extend(utaut_scores[:,2])
        FC_list.extend(utaut_scores[:,3])

    return sentiments, PE_list, EE_list, SI_list, FC_list

In [7]:
import re
import emoji

slang_dict = {
    "ga": "tidak",
    "gak": "tidak",
    "gk": "tidak",
    "nggak": "tidak",
    "tp": "tapi",
    "jd": "jadi",
    "bgt": "banget",
    "yg": "yang",
    "krn": "karena"
}

def normalize_slang(text):
    words = text.split()
    return " ".join([slang_dict.get(w, w) for w in words])


def preprocess_text(text):
    text = str(text)

    # 1. Lowercase
    text = text.lower()

    # 2. Remove URL
    text = re.sub(r'http\S+|www\S+', ' ', text)

    # 3. Remove mention
    text = re.sub(r'@\w+', ' ', text)

    # 4. Remove hashtag symbol only (kata tetap)
    text = re.sub(r'#', '', text)

    # 5. Remove emoji
    text = emoji.replace_emoji(text, replace='')

    # 6. Slang normalization ringan
    text = normalize_slang(text)

    # 7. Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()

    return text


In [8]:
df = pd.read_csv("datasets_final.csv", sep=";", encoding="utf-8-sig")
df["comment"] = df["comment"].astype(str).apply(preprocess_text)
df["comment"] = df["comment"].fillna("").astype(str)
texts = df["comment"].tolist()

In [9]:
sentiments,PE,EE,SI,FC = batch_predict(texts,batch_size=32)

100%|██████████| 349/349 [15:49<00:00,  2.72s/it]


In [10]:
df["sentiment"] = sentiments

df["PE"] = PE
df["EE"] = EE
df["SI"] = SI
df["FC"] = FC

In [11]:
TAM_LABELS = ["PE","EE","SI","FC"]

if all(label in df.columns for label in TAM_LABELS):

    y_true = df[TAM_LABELS].values

    y_pred = df[["PE","EE","SI","FC"]].values

    # MAE
    mae = mean_absolute_error(y_true,y_pred)

    # RMSE
    rmse = np.sqrt(mean_squared_error(y_true,y_pred))

    # Exact accuracy
    exact_acc = (y_true == y_pred).all(axis=1).mean()

    # Accuracy ±1
    tol_acc = (np.abs(y_true - y_pred) <= 1).all(axis=1).mean()

    print("MAE:",mae)
    print("RMSE:",rmse)
    print("Exact Accuracy:",exact_acc)
    print("Accuracy ±1:",tol_acc)

MAE: 0.0
RMSE: 0.0
Exact Accuracy: 1.0
Accuracy ±1: 1.0


In [12]:
labels = ["PE","EE","SI","FC"]

dim_results = {}

for label in labels:

    mae_dim = mean_absolute_error(df[label],df[label])

    dim_results[label] = mae_dim

In [13]:
evaluation = pd.DataFrame({

"model":["IndoBERT"],

"MAE":[mae],

"RMSE":[rmse],

"ExactAccuracy":[exact_acc],

"AccuracyTolerance":[tol_acc],

"PE_MAE":[mean_absolute_error(y_true[:,0],y_pred[:,0])],

"EE_MAE":[mean_absolute_error(y_true[:,1],y_pred[:,1])],

"SI_MAE":[mean_absolute_error(y_true[:,2],y_pred[:,2])],

"FC_MAE":[mean_absolute_error(y_true[:,3],y_pred[:,3])]

})

evaluation.to_csv("evaluation_indobert.csv",index=False)

In [14]:
df.to_csv(
    "outputs_indobert.csv",
    index=False
)